# Figure 2d: Paired expression comparison

This notebook reproduces Figure 2d by comparing normalized expression between paired tumor and normal samples.

## Workflow

1. Load sample metadata and normalized expression values.
2. Align expression columns with metadata by sample name.
3. Prepare a tidy plotting table.
4. Draw paired violin, point, line, and box plots for the selected gene.
5. Save the figure as a PDF.

All file paths and plotting parameters are defined in the configuration cell below.

In [ ]:
from pathlib import Path

import pandas as pd
from plotnine import (
    aes,
    element_text,
    geom_boxplot,
    geom_line,
    geom_point,
    geom_violin,
    ggplot,
    labs,
    scale_color_manual,
    scale_fill_manual,
    theme,
    theme_classic,
)


## Configuration

Update these paths when running the notebook in a different environment. The target feature is the Ensembl transcript identifier used in the original analysis.

In [ ]:
project_dir = Path('/data1/DYY/bambu')
metadata_file = project_dir / '288_group.csv'
expression_file = project_dir / '288_FDR' / 'all_gene.csv'
output_file = project_dir / 'plot' / 'figure2' / '2d_down.pdf'

target_feature = 'ENSG00000167747.17'
feature_label = 'C19orf48'

condition_order = ['Tumor', 'Normal']
condition_colors = {
    'Tumor': '#FF5051',
    'Normal': '#CAE8F2',
}

sample_colors = [
    '#bcbd22', '#7f7f7f', '#9467bd', '#1f77b4', '#8c564b',
    '#17becf', '#e377c2', '#d62728', '#2ca02c', '#ff7f0e',
]


## Helper functions

In [ ]:
def validate_columns(dataframe, required_columns, table_name):
    """Raise an informative error when required columns are absent."""
    missing_columns = sorted(set(required_columns) - set(dataframe.columns))
    if missing_columns:
        raise ValueError(
            f'{table_name} is missing required columns: {missing_columns}'
        )


def prepare_plot_data(metadata_file, expression_file, target_feature):
    """Load, validate, align, and reshape data for one target feature."""
    metadata = pd.read_csv(metadata_file, sep='\t')
    validate_columns(metadata, {'name', 'Type', 'Sample ID'}, 'Metadata')

    if metadata['name'].duplicated().any():
        duplicated_names = metadata.loc[
            metadata['name'].duplicated(), 'name'
        ].tolist()
        raise ValueError(
            f'Metadata contains duplicated sample names: {duplicated_names[:5]}'
        )

    expression_table = pd.read_csv(expression_file, sep='\t')
    expression_table = expression_table.drop(
        columns=['p_adjust', 'P_value'], errors='ignore'
    )

    if target_feature not in expression_table.columns:
        raise KeyError(
            f'Target feature {target_feature!r} was not found in {expression_file}.'
        )

    expression_by_sample = (
        expression_table
        .set_index(expression_table.columns[0])
        .T
        .rename_axis('name')
        .reset_index()
    )

    plot_data = metadata.merge(
        expression_by_sample[['name', target_feature]],
        on='name',
        how='inner',
        validate='one_to_one',
    )

    if len(plot_data) != len(metadata):
        missing_samples = sorted(set(metadata['name']) - set(plot_data['name']))
        raise ValueError(
            'Some metadata samples were not found in the expression matrix: '
            f'{missing_samples[:10]}'
        )

    plot_data[target_feature] = pd.to_numeric(
        plot_data[target_feature], errors='raise'
    )
    plot_data['sample_group'] = plot_data['Sample ID'].str[:3]
    plot_data['Type'] = pd.Categorical(
        plot_data['Type'], categories=condition_order, ordered=True
    )
    plot_data = plot_data.sort_values(['Sample ID', 'Type']).reset_index(drop=True)

    return plot_data


def create_paired_expression_plot(
    plot_data,
    target_feature,
    feature_label,
    condition_colors,
    sample_colors,
):
    """Create the paired violin plot used in Figure 2d."""
    sample_groups = plot_data['sample_group'].drop_duplicates().tolist()
    if len(sample_groups) > len(sample_colors):
        raise ValueError(
            f'{len(sample_groups)} sample groups were found, but only '
            f'{len(sample_colors)} colors were supplied.'
        )

    sample_color_map = dict(zip(sample_groups, sample_colors))

    figure = (
        ggplot(plot_data, aes(x='Type', y=target_feature))
        + geom_violin(
            aes(fill='Type'),
            style='left-right',
            draw_quantiles=[0.25, 0.50, 0.75],
            alpha=0.60,
            size=0.50,
            color='#57606f',
        )
        + geom_line(
            aes(group='Sample ID', color='sample_group'),
            size=0.50,
            alpha=0.60,
        )
        + geom_point(
            aes(group='Sample ID', color='sample_group'),
            size=0.60,
            alpha=0.60,
        )
        + geom_boxplot(
            aes(fill='Type'),
            width=0.10,
            alpha=0.60,
            size=0.50,
            color='#57606f',
            show_legend=False,
            outlier_alpha=0,
        )
        + scale_fill_manual(values=condition_colors)
        + scale_color_manual(values=sample_color_map)
        + labs(
            x=None,
            y='Normalized expression',
            title=feature_label,
            fill='Condition',
            color='Sample group',
        )
        + theme_classic()
        + theme(
            figure_size=(8, 6),
            plot_title=element_text(ha='center'),
        )
    )

    return figure


## Load and validate data

In [ ]:
plot_data = prepare_plot_data(
    metadata_file=metadata_file,
    expression_file=expression_file,
    target_feature=target_feature,
)

print(f'Number of samples: {len(plot_data)}')
print(f'Number of paired sample IDs: {plot_data["Sample ID"].nunique()}')
print(plot_data['Type'].value_counts(sort=False))


## Generate and save Figure 2d

In [ ]:
figure_2d = create_paired_expression_plot(
    plot_data=plot_data,
    target_feature=target_feature,
    feature_label=feature_label,
    condition_colors=condition_colors,
    sample_colors=sample_colors,
)

output_file.parent.mkdir(parents=True, exist_ok=True)
figure_2d.save(output_file, dpi=400, verbose=False)

print(f'Figure saved to: {output_file}')
figure_2d
